In [27]:
import boto3
import json
import os
from datetime import datetime
import time
from typing import Dict, List, Tuple
import re

# Bedrock client for batch inference job
bedrock = boto3.client(service_name="bedrock",region_name="eu-west-1")

# Create an S3 client
s3 = boto3.client('s3',region_name="eu-west-1")

# # Set the S3 bucket name and prefix for the text files
bucket_name = 'mara-batch-inference'
# raw_data_prefix = 'virat'
output_prefix = 'output-data'

# # Batch API parameters:
roleArn = "arn:aws:iam::774305600887:role/mara-batch-inference-role"  # arn of role (mara-inference-policy)
model_input_summary_prefix = 'input-data'
jobName = 'batch-job-ga' + str(int(datetime.now().timestamp()))
model_id = 'anthropic.claude-3-haiku-20240307-v1:0' # or use other model
# model_id = "eu.anthropic.claude-3-5-sonnet-20240620-v1:0"

# # Configure logging
# logger = logging.getLogger()
# logger.setLevel(logging.INFO)

# Get AWS region
AWS_REGION = os.environ['AWS_REGION']

# Initialize AWS clients
dynamodb = boto3.resource('dynamodb')
questions_table = dynamodb.Table(f'MaraQuestions-{AWS_REGION}')
risk_assessment_table = dynamodb.Table(f'ClientRiskAssessment-{AWS_REGION}')

filename = f'batch-{int(datetime.now().timestamp())}.jsonl'

In [2]:
inputDataConfig=({
    "s3InputDataConfig": {
        "s3Uri": f"s3://{bucket_name}/{model_input_summary_prefix}/{filename}"
    }
})

outputDataConfig=({
    "s3OutputDataConfig": {
        "s3Uri": f"s3://{bucket_name}/{model_input_summary_prefix}/{output_prefix}/"
    }
})

In [3]:
message_body ={
    "requestId": "efcc832b-4dc0-4b50-b1f4-576819585518",
    "firmName": "Jos' Paellas",
    "output": "s3://mara-ingestion-eu-west-1-774305600887/processed/efcc832b-4dc0-4b50-b1f4-576819585518/results.json",
    "aumSize": "$100bn+"
}


In [4]:
# Extract and validate required fields
base_request_id = message_body.get('requestId')
firm_name = message_body.get('firmName')
s3_output = message_body.get('output')

In [5]:
def get_json_from_s3(s3_uri: str) -> Dict:
    """Load JSON content from S3"""
    try:
        # Parse S3 URI to get bucket and key
        _, path = s3_uri.replace("s3://", "").split("/", 1)
        bucket = s3_uri.split("/")[2]
        
        # Get object from S3
        response = s3.get_object(Bucket=bucket, Key=path)
        content = json.loads(response['Body'].read().decode('utf-8'))
        # logger.info(f"Successfully loaded JSON from {s3_uri}")
        print(f"Successfully loaded JSON from {s3_uri}")
        return content
    except Exception as e:
        # logger.error(f"Error loading JSON from S3: {str(e)}")
        print(f"Error loading JSON from S3: {str(e)}")
        raise


In [6]:
def get_all_questions() -> List[Dict]:
    """Retrieve all questions from DynamoDB"""
    try:
        # Initialize list to store all questions
        questions = []
        
        # Initial scan without any filter
        response = questions_table.scan()
        
        # Process initial response
        for item in response.get('Items', []):
            if 'questionId' in item:  # Ensure it has required fields
                questions.append(item)
        
        # Handle pagination
        while 'LastEvaluatedKey' in response:
            response = questions_table.scan(
                ExclusiveStartKey=response['LastEvaluatedKey']
            )
            for item in response.get('Items', []):
                if 'questionId' in item:
                    questions.append(item)
        
        # Sort by questionId
        questions.sort(key=lambda x: x['questionId'])
        
        print((f"Retrieved total {len(questions)} questions"))
        # logger.info(f"Retrieved total {len(questions)} questions")
        
        # Debug output for first few questions
        if questions:
            # logger.info("Sample questions found:")
            print("Sample questions found:")
            for q in questions[:2]:  # Show first 2 questions
                # logger.info(f"ID: {q['questionId']}, Title: {q.get('sectionTitle', 'N/A')}")
                print(f"ID: {q['questionId']}, Title: {q.get('sectionTitle', 'N/A')}")
        else:
            print("No questions found in the table")
            print("Available fields in table:")
            # logger.warning("No questions found in the table")
            # logger.info("Available fields in table:")
            # Get a sample item to show available fields
            sample = questions_table.scan(Limit=1)
            if sample.get('Items'):
                # logger.info(f"Sample item fields: {list(sample['Items'][0].keys())}")
                print(f"Sample item fields: {list(sample['Items'][0].keys())}")
        
        return questions
        
    except Exception as e:
        print(f"Error querying DynamoDB: {str(e)}")
        print("Stack trace:", exc_info=True)
        # logger.error(f"Error querying DynamoDB: {str(e)}")
        # logger.error("Stack trace:", exc_info=True)
        raise

In [7]:
# text = """Virat Kohli, born on November 5, 1988, in Delhi, India, is one of the most celebrated cricketers in the world. Known for his aggressive batting style and exceptional leadership, Kohli has earned nicknames like "King Kohli," "Chase Master," and "Run Machine"1. He has played a pivotal role in numerous victories for the Indian cricket team, including the 2011 ODI World Cup and the 2024 T20 World Cup1.

# Kohli's career is marked by numerous records and accolades. He is the highest run-scorer in the Indian Premier League (IPL) and has the most centuries in ODIs1. His dedication to fitness and consistency has set new standards in Indian cricket, making him a role model for aspiring cricketers2.

# Off the field, Kohli is known for his philanthropic efforts through the Virat Kohli Foundation, which focuses on supporting underprivileged children and promoting sports1. He is married to Bollywood actress Anushka Sharma, and together they are one of the most influential couples in India1."""

In [8]:
def build_advanced_mara_prompt(content: Dict, question: List[Dict], section_title: str, 
                        max_prompt_length: int = 70000) -> str:
    """
    Build a strictly factual prompt for Claude to analyze MARA regulatory content without any subjective assessments.
    
    Args:
        content: Dictionary containing regulatory content to analyze (raw_data from results.json)
        questions: List of question dictionaries, each with at least a 'text' key
        section_title: Title of the regulatory section being analyzed (e.g. "Algorithm Trading Risk")
        max_prompt_length: Maximum allowed prompt length in characters
        
    Returns:
        Formatted prompt string for Claude analysis
    """
    # Input validation
    if not isinstance(content, dict):
        raise ValueError("Content must be a dictionary")
    
    # Generate enhanced prompt with comprehensive analysis framework
    prompt = f"""You are conducting a Market Abuse Risk Assessment (MARA) for the "{section_title}" category as a financial compliance expert. Your task is to analyze the provided content and answer specific questions with STRICTLY FACTUAL responses. You must NEVER include ANY subjective assessments or qualitative judgments about the quality, robustness, or effectiveness of controls.

## MARA ANALYSIS METHODOLOGY:
Follow this multi-pass analysis process for each question:

### FIRST PASS: FACT EXTRACTION AND APPLICABILITY ASSESSMENT
1. Identify all entries in the content related to {section_title} by:
   - Filtering for exact risk area matches
   - Searching for relevant keywords across all entries
   - Identifying related subtopics that may contain relevant information

2. Determine applicability based on business context:
   - Review business model information (asset classes, trading activities, client types)
   - Check for explicit statements about applicability in the documentation
   - Look for statements that the activity doesn't occur or isn't relevant
   - Consider whether the business structure inherently excludes the risk

3. For YES/NO responses, document their factual meaning:
   - "YES" indicates presence of a control, process, or activity
   - "NO" indicates absence of a control, process, or activity
   - "N/A" indicates non-applicability to business model
   - "Not applicable (0%)" responses strongly indicate non-applicability

4. Extract factual information from explanatory text:
   - Implementation details
   - Procedural steps
   - Tools or systems 
   - Timeframes or frequencies

### SECOND PASS: INFORMATION ORGANIZATION
1. Look beyond the primary risk area for additional factual information:
   - Related risk areas with objective connections to the question
   - Enterprise-wide processes objectively related to this specific area
   - Business model factual information that provides context

2. Catalog evidence according to this factual hierarchy:
   - Direct statements of fact about controls or processes
   - Explicitly referenced documents, systems, or procedures
   - Factual descriptions of activities or responsibilities
   - Objective data points from related areas

3. Document information gaps objectively:
   - Questions without corresponding answers
   - Partial or incomplete information
   - Explicitly contradictory statements
   - Areas where factual details are missing

### THIRD PASS: COMPLIANCE ASSESSMENT
1. Determine the appropriate response type with a focus on identifying non-applicability:
   - NOT_APPLICABLE (0%): ONLY when there is explicit evidence that the question doesn't apply to the business model, such as explicit "Not applicable (0%)" responses or clear statements that the activity is not relevant to the firm's business model
   - NULL (25-100%): When NO evidence exists for the question (neither about presence nor absence of controls)
   - REAL (25-100%): When ANY evidence exists, even if that evidence indicates that controls are missing or not implemented

2. Assign an applicability percentage based on business context:
   - Default: 100% unless evidence indicates lower applicability
   - 0%: For NOT_APPLICABLE questions (ONLY when there is explicit evidence that the risk doesn't apply to the business model)
   - 25%: When business activities indicate low probability/exposure (0-25%)
   - 50%: When business activities indicate moderate probability/exposure (>25-50%)
   - 75%: When business activities indicate significant probability/exposure (>50-75%)
   - 100%: When business activities indicate high probability/exposure (>75%) OR core activity

3. Present information in the voice of a compliance officer describing their firm's approach

## RESPONSE TYPE CLARIFICATION
Critical distinction for response types:
- NOT_APPLICABLE (0%): Use ONLY when explicit evidence shows the question is irrelevant to the business model (e.g., "Not applicable (0%)" responses, statements about business activities that make the risk irrelevant)
- REAL: Use whenever ANY evidence exists, EVEN IF that evidence indicates the absence of controls
- NULL: Use ONLY when NO evidence can be found, neither confirming nor denying the presence of controls

Examples:
- If evidence says "NO, we don't have such controls" → REAL response (not NOT_APPLICABLE)
- If evidence says "Not applicable (0%)" or "We don't trade these instruments" → NOT_APPLICABLE
- If no evidence is found at all → NULL

## AVOID SUBJECTIVE LANGUAGE
Avoid these types of subjective language:
- Value judgments: "good," "bad," "strong," "weak," "robust," "insufficient," etc.
- Effectiveness assessments: "effective," "adequate," "comprehensive," "limited," etc.
- Quality descriptions: "high-quality," "thorough," "detailed," "extensive," etc.
- Achievement language: "successfully," "effectively," "well-designed," etc.
- Risk evaluation: "risky," "safe," "dangerous," "concerning," etc.

## COMPLIANCE OFFICER VOICE REQUIREMENTS:
- Write as a compliance officer describing your firm's approach
- Use first-person plural pronouns ("we," "our") when describing the firm's activities
- Use factual, straightforward language to describe controls and processes
- Present information directly without excessive qualifiers
- Use active voice when describing what the firm does: "We have," "We use," "We conduct"
- Be factual without making qualitative judgments about effectiveness
- Acknowledge gaps in controlled, factual language
- Clearly state when activities don't apply to the business model
- Maintain professional, matter-of-fact tone throughout

## EVIDENCE DOCUMENTATION:
- Quote ALL relevant evidence verbatim with exact attribution
- Present multiple pieces of evidence when available
- Include conflicting evidence if found
- Be particularly attentive to evidence of non-applicability (e.g., "Not applicable (0%)" responses)
- Look for business model information that establishes context for applicability
- When considering non-applicability, examine asset classes, client types, and trading activities

## STRICTLY FACTUAL EXAMPLE:
Question: "How do your controls manage algorithmic trading risks?"

<evidence>
From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use your own in-house algorithms to execute orders?", NBEL Answer: "NO"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use the execution algos of your third-party brokers?", NBEL Answer: "NO TBC - Could happen if doing DMA on ICE/Euqrex"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use your own in-house investment decision making algos?", NBEL Answer: "NO"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use third-party investment decision making algos?", NBEL Answer: "NO"
</evidence>

<answer>
REAL (25%). We do not use algorithmic trading as part of our business operations.

We do not use in-house algorithms for execution or investment decision-making, nor do we generally use third-party algorithms. Our approach focuses on non-algorithmic trading methods. The only potential exception is our limited use of Direct Market Access (DMA) on specific exchanges (ICE/Eurex), which might involve some interaction with third-party algorithms in certain circumstances. We do not have specific algorithmic trading controls in place because our business model generally excludes this activity.

The documentation shows "NO" responses to all questions about algorithm usage. Our response "NO TBC - Could happen if doing DMA on ICE/Euqrex" indicates the limited potential for algorithmic exposure through specific DMA channels, which explains the 25% applicability score. The information available does not include details about any specific controls for this limited exposure case.
</answer>

## CONTENT TO ANALYZE:"""

    # Safe content serialization with error handling
    try:
        # Serialize content as formatted JSON
        content_json = json.dumps(content, indent=2)
        
        # Content truncation if needed
        if len(content_json) > max_prompt_length // 2:
            # Strategic truncation that prioritizes the relevant section
            # First attempt to isolate just the relevant section
            section_data = []
            if 'raw_data' in content:
                section_data = [item for item in content['raw_data'] 
                                if section_title.lower() in item.get('data', {}).get('Risk Area', '').lower()]
                
            # If we have section-specific data and it's smaller than our limit, use just that
            if section_data and len(json.dumps(section_data, indent=2)) < max_prompt_length // 2:
                content_json = json.dumps({"raw_data": section_data}, indent=2)
            else:
                # Otherwise use standard truncation
                content_json = content_json[:max_prompt_length // 2] + "\n...[Content truncated for length]..."
    except Exception as e:
        raise ValueError(f"Failed to serialize content: {str(e)}")
        
    # Complete the prompt with content and questions
    prompt += f"""
{content_json}

## QUESTIONS FOR {section_title.upper()}:
{question}

## REQUIRED ANSWER FORMAT:
For each question, follow this EXACT structure with XML tags:

<question>
[Question number]. [Question text exactly as provided]
</question>

<evidence>
[ALL relevant evidence with attribution in this format:]
From Risk Area "[risk area]", Question "[question]", Subquestion "[subquestion]", NBEL Answer: "[answer]"

[Include multiple evidence pieces, each with proper attribution]
</evidence>

<answer>
REAL (75%). [One-sentence factual statement of what exists or doesn't exist]

[Factual information paragraph - ONLY list what objectively exists or doesn't exist, with NO qualitative assessment]

[Evidence summary paragraph - ONLY list what evidence was found, with NO interpretation of its meaning, quality, or implications]
</answer>

CRITICAL FORMAT REQUIREMENTS:
1. Use EXACT XML tags as shown: <question>, </question>, <evidence>, </evidence>, <answer>, </answer>
2. Each tag must appear on its own line with no extra text
3. Response type must be one of: REAL, NULL, or NOT_APPLICABLE (all caps)
4. Score must be a number: 0, 25, 50, 75, or 100 (include the % symbol)
5. For each piece of evidence, include the exact Risk Area, Question, Subquestion, and NBEL Answer
6. CONTAIN ONLY FACTUAL INFORMATION - avoid ALL adjectives, adverbs, and other modifiers that imply quality or effectiveness
7. DO NOT use ANY words that suggest quality, effectiveness, capability, sufficiency, or appropriateness
8. Use passive voice when possible to avoid attributing agency or intent

CRITICALLY IMPORTANT: Use REAL for ANY evidence (even if it shows lack of controls), NOT_APPLICABLE ONLY for explicit evidence of non-applicability to business model, and NULL ONLY when no evidence exists at all.

STRICTLY FACTUAL OUTPUT FORMAT EXAMPLE:
<question>
1. How do your controls manage algorithmic trading risks?
</question>

<evidence>
From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use your own in-house algorithms to execute orders?", NBEL Answer: "NO"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use the execution algos of your third-party brokers?", NBEL Answer: "NO TBC - Could happen if doing DMA on ICE/Euqrex"
</evidence>

<answer>
REAL (25%). We do not use algorithmic trading in our business operations.

We do not use in-house algorithms for execution or investment decision-making, nor do we typically employ third-party algorithms. This is a deliberate aspect of our business model. The only exception involves potential use of Direct Market Access (DMA) on ICE/Eurex exchanges, which might occasionally interact with third-party algorithms. We don't maintain specific algorithmic trading controls since this activity falls outside our standard operations.

Our responses consistently show "NO" to all algorithm usage questions across various categories. The response "NO TBC - Could happen if doing DMA on ICE/Euqrex" indicates our limited potential exposure to algorithmic trading through specific exchange access, which explains the 25% applicability score. We have not implemented specific controls, monitoring, or testing frameworks for algorithmic trading given its minimal relevance to our business model.
</answer>
"""

    # Verify final prompt length
    if len(prompt) > max_prompt_length:
        raise ValueError(f"Generated prompt exceeds maximum length of {max_prompt_length} characters")
        
    return prompt

In [9]:

def prepare_model_inputs(text, questions):
    # Initialize the model_inputs list
    model_inputs = []
    # Process each text file
    for question in questions:
        questions_text = question
        section_title = question.get("sectionTitle"," ")
        # Generate a unique record ID
        record_id = str(int(datetime.now().timestamp())) # you can replace this with your own logic
        # Prepare the input text for the Anthropic API
        prompt= build_advanced_mara_prompt(text,questions_text,section_title)
        # Define the request body for the Anthropic API (if you change the model you have to use the body of that model)
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "messages": [{"role": 'user',
                           "content": [
                               {'type': 'text',
                                'text': prompt}]
                           }],
            "max_tokens": 4096,
            "temperature": 0.1,
        }
        # Prepare the model input
        model_input = {
            "recordId": record_id,
            "modelInput": body
        }
        # Append the model input to the list
        model_inputs.append(model_input)
    return model_inputs

def upload_jsonl_to_s3(data, s3_client, bucket_name,filename, bucket_subfolder=None):
    """
    Upload JSONL data directly to S3 without saving locally
    
    Args:
    - data (list): List of dictionaries to be converted to JSONL
    - s3_client (boto3.client): Configured S3 client
    - bucket_name (str): Name of the S3 bucket
    - bucket_subfolder (str, optional): Subfolder path in the bucket
    
    Returns:
    - bool: True if upload successful, False otherwise
    """
    try:
        # Generate filename with timestamp
        # filename = f'batch-{int(datetime.now().timestamp())}.jsonl'
        
        # Convert data to JSONL string
        jsonl_content = '\n'.join(json.dumps(item) for item in data)
        
        # Prepare the full S3 object key
        object_key = filename if bucket_subfolder is None else f"{bucket_subfolder}/{filename}"
        
        # Upload directly to S3
        s3_client.put_object(
            Bucket=bucket_name, 
            Key=object_key, 
            Body=jsonl_content.encode('utf-8')
        )
        # print()
        print(f"Successfully uploaded {filename} to {bucket_name}/{object_key}")
        return True
    
    except Exception as e:
        print(f"Error uploading to S3: {e}")
        return False


In [10]:
# Get content and section titles
json_content = get_json_from_s3(s3_output)
questions = get_all_questions()

Successfully loaded JSON from s3://mara-ingestion-eu-west-1-774305600887/processed/efcc832b-4dc0-4b50-b1f4-576819585518/results.json
Retrieved total 112 questions
Sample questions found:
ID: AT-0037-0, Title: Algorithm Trading
ID: AT-0038-0, Title: Algorithm Trading


In [11]:
# Prepare and upload model inputs directly to S3
model_input_jsonl = prepare_model_inputs(json_content, questions)
upload_jsonl_to_s3(
    data=model_input_jsonl, 
    s3_client=s3, 
    bucket_name=bucket_name, 
    filename=filename,
    bucket_subfolder=model_input_summary_prefix
)

Successfully uploaded batch-1742315506.jsonl to mara-batch-inference/input-data/batch-1742315506.jsonl


True

In [12]:
response=bedrock.create_model_invocation_job(
    roleArn=roleArn,
    modelId=model_id,
    jobName=jobName,
    inputDataConfig=inputDataConfig,
    outputDataConfig=outputDataConfig
)

In [13]:
jobArn = response.get('jobArn')
job_id = jobArn.split('/')[1]

print(jobArn)

status = ''
while status not in ['Completed', 'Failed']:
    job_response = bedrock.get_model_invocation_job(jobIdentifier=jobArn)
    status = job_response['status']
    if status == 'Failed':
        print(job_response)
    elif status == 'Completed':
        print(datetime.now(), ": ", status)
        break
    else: 
        print(datetime.now(), ": ", status)
        time.sleep(60)

arn:aws:bedrock:eu-west-1:774305600887:model-invocation-job/gmeu34kd32or
2025-03-18 16:32:04.088288 :  Submitted
2025-03-18 16:33:04.174379 :  Validating
2025-03-18 16:34:04.260060 :  Validating
2025-03-18 16:35:04.362908 :  Validating
2025-03-18 16:36:04.466157 :  Scheduled
2025-03-18 16:37:04.576050 :  InProgress
2025-03-18 16:38:04.697925 :  InProgress
2025-03-18 16:39:04.812516 :  InProgress
2025-03-18 16:40:04.930658 :  InProgress
2025-03-18 16:41:05.078810 :  Completed


In [19]:
import boto3
import json
# Create an S3 client
s3 = boto3.client('s3',region_name="eu-west-1")

# Set the S3 bucket name and prefix for the text files. 
job_id = "oj42ie90ezca"
filename = "batch-1742302706.jsonl"
# Set the S3 bucket name and prefix for the text files
bucket_name = 'mara-batch-inference'
raw_data_prefix = 'virat'
output_prefix = 'output-data'

model_input_summary_prefix = 'input-data'

# Last part in the path is the batch job's job id
prefix = f"{model_input_summary_prefix}/{output_prefix}/{job_id}/"

# Initialize the list
output_data = []

# Read the JSON file from S3
try:
    object_key = f"{prefix}{filename}.out"
    response = s3.get_object(Bucket=bucket_name, Key=object_key)
    json_data = response['Body'].read().decode('utf-8')

    # Process the JSON data
    for line in json_data.splitlines():
        data = json.loads(line)
        output_entry = {
            'request_id': data['recordId'],
            'output_text': data['modelOutput']['content'][0]['text'],
            'observability': {
                'input_tokens': data['modelOutput']['usage']['input_tokens'],
                'output_tokens': data['modelOutput']['usage']['output_tokens'],
                'model': data['modelOutput']['model'],
                'stop_reason': data['modelOutput']['stop_reason'],
                'request_id': data['recordId'],
                'max_tokens': data['modelInput']['max_tokens'],
                'temperature': data['modelInput']['temperature']
            }
        }
        output_data.append(output_entry)
    print(f"Successfully read {len(output_data)} JSON objects from S3.")
except Exception as e:
    print(f"Error reading JSON file from S3: {e}")


Successfully read 112 JSON objects from S3.


In [20]:
def parse_answer(analysis: str, question: Dict, request_id: str) -> Tuple[str, str, str, str, str]:
    """
    Parse response for a single question with improved robustness
    
    Args:
        analysis (str): The analysis text to parse
        question (Dict): Dictionary containing question details
        request_id (str): Unique identifier for the request
    
    Returns:
        Tuple containing:
        - request_id
        - question_id
        - formatted answer
        - evidence
        - section title
    """
    # logger.info(f"Starting to parse response for question ID {question['questionId']}")
    print(f"Starting to parse response for question ID {question['questionId']}")
    
    # Extract question text for reference
    question_text = question['text']
    question_id = question['questionId']
    section_title = question.get('sectionTitle', 'Unknown Section')
    
    # Try to extract content using XML tags first
    full_blocks = re.findall(r'<question>(.*?)</question>\s*<evidence>(.*?)</evidence>\s*<answer>(.*?)</answer>', analysis, re.DOTALL)
    
    # Prepare variables for evidence and answer
    evidence = ""
    answer = ""
    
    # Extract XML blocks if found
    if full_blocks:
        _, evidence, answer = full_blocks[0]
    
    # If XML extraction fails, try alternative parsing methods
    if not evidence or not answer:
        # Look for evidence-like content
        evidence_candidates = re.findall(r'(?:Risk Area|From Risk Area|NBEL Answer)[^\n]+', analysis, re.DOTALL)
        if evidence_candidates:
            evidence = "\n".join(evidence_candidates)
        
        # Look for answer-like content with REAL/NULL/NOT_APPLICABLE
        answer_match = re.search(r'(REAL|NULL|NOT_APPLICABLE).*?(\d+%|\(\d+%\))', analysis, re.DOTALL)
        if answer_match:
            answer = answer_match.group(0) + analysis[answer_match.end():]
    
    # Clean up evidence and answer
    evidence = re.sub(r'</?evidence>', '', evidence).strip()
    answer = re.sub(r'</?answer>', '', answer).strip()
    
    # Ensure meaningful content exists
    if not evidence:
        evidence_lines = re.findall(r'(?:Risk Area|From Risk Area|NBEL Answer)[^\n]+', analysis, re.DOTALL)
        evidence = "\n".join(evidence_lines) if evidence_lines else "No structured evidence format found"
    
    if not answer:
        answer_match = re.search(r'(REAL|NULL|NOT_APPLICABLE).*?(\d+%|\(\d+%\))', analysis, re.DOTALL)
        if answer_match:
            answer = answer_match.group(0) + analysis[answer_match.end():]
        else:
            # Create a basic answer using the question text
            answer = f"REAL (50%). Analysis of evidence for: {question_text}"
    
    # Standardize answer format
    if not re.search(r'(REAL|NULL|NOT_APPLICABLE)\s*\(?\d+%\)?', answer):
        # Determine response type and score
        if 'not applicable' in answer.lower():
            response_type = "NOT_APPLICABLE"
            score = "0"
        elif 'null' in answer.lower():
            response_type = "NULL"
            score = "100"
        else:
            response_type = "REAL"
            score = "50"
        
        # Format the answer
        answer = f"{response_type} ({score}%). {answer}"
    
    # Create 3-paragraph structure if needed
    if answer and not re.search(r'\n\n', answer):
        sentences = re.split(r'(?<=[.!?])\s+', answer)
        if len(sentences) >= 3:
            mid_point = len(sentences) // 3
            p1 = ' '.join(sentences[:mid_point])
            p2 = ' '.join(sentences[mid_point:2*mid_point])
            p3 = ' '.join(sentences[2*mid_point:])
            answer = f"{p1}\n\n{p2}\n\n{p3}"
    
    # If no meaningful content found, create a NULL response
    if not evidence.strip() or not answer.strip():
        # logger.warning(f"Missing evidence or answer for question {question_id}. Creating NULL response.")
        print(f"Missing evidence or answer for question {question_id}. Creating NULL response.")
        return (
            request_id,
            question_id,
            "NULL RESPONSE (Applicability Score: 100%)\n\nRequired information is missing from the provided content.\n\nNo relevant evidence was found after searching through all available documentation.",
            "No evidence available",
            section_title
        )
    
    # Return the parsed answer
    return (
        request_id,
        question_id,
        answer.strip(),
        evidence.strip(),
        section_title
    )

In [21]:
def update_risk_assessment_with_answers(request_id: str, firm_name: str, question_id: str, 
                                      question_text: str, answer: str, evidence: str, section_title: str):
    """Update risk assessment with validation and proper error handling"""
    # logger.info(f"Updating DynamoDB with evidence: {evidence}")
    print(f"Updating DynamoDB with evidence: {evidence}")
    try:
        # Validate firm_name to prevent GSI errors
        if not firm_name or not isinstance(firm_name, str):
            print(f"Invalid firm_name for request {request_id}: {firm_name}")
            # logger.error(f"Invalid firm_name for request {request_id}: {firm_name}")
            raise ValueError("firm_name must be a non-empty string")

        update_expression = "SET firmName = :firm_name, answer = :answer, evidence = :evidence, sectionTitle = :section_title, question = :question_text, lastUpdated = :timestamp"
        
        # First, get the current item
        current_item = risk_assessment_table.get_item(
            Key={
                'requestId': request_id,
                'questionId': question_id
            }
        ).get('Item', {})
        
        # logger.info(f"Current item before update: {current_item}")
        print(f"Current item before update: {current_item}")

        expression_values = {
            ':firm_name': firm_name,
            ':answer': answer,
            ':evidence': evidence,
            ':section_title': section_title,
            ':question_text': question_text,
            ':timestamp': time.strftime('%Y-%m-%d %H:%M')
        }

        # logger.info(f"Update Expression: {update_expression}")
        # logger.info(f"Expression Values: {expression_values}")
        print(f"Update Expression: {update_expression}")
        print(f"Expression Values: {expression_values}")
        
        response = risk_assessment_table.update_item(
            Key={
                'requestId': request_id,
                'questionId': question_id
            },
            UpdateExpression=update_expression,
            ExpressionAttributeValues=expression_values,
            ReturnValues='ALL_NEW'
        )

        # logger.info(f"Item after update: {response.get('Attributes', {})}")
        # logger.info(f"Successfully updated risk assessment for request {request_id}, question {question_id}")
        print(f"Item after update: {response.get('Attributes', {})}")
        print(f"Successfully updated risk assessment for request {request_id}, question {question_id}")
        return response
        
    except ValueError as ve:
        print(f"Validation error for {request_id}: {str(ve)}")
        # logger.error(f"Validation error for {request_id}: {str(ve)}")
        return None
        
    except ClientError as e:
        error_code = e.response.get('Error', {}).get('Code', '')
        if error_code == 'ValidationException':
            print(f"DynamoDB validation error for {request_id}: {str(e)}")
            # logger.error(f"DynamoDB validation error for {request_id}: {str(e)}")
            return None
        else:
            logger.error(f"DynamoDB error for {request_id}: {str(e)}")
            raise


In [24]:
len(output_data)

112

In [29]:
answers = []
for i in range(len(output_data)):
    analysis = output_data[i]["output_text"]
    question_text = questions[i]
    answer = parse_answer(analysis, question_text, base_request_id)
    answers.append(answer)
    print(questions[i])
    print("80"*100)
    print(analysis)
    print("80"*100)

Starting to parse response for question ID AT-0037-0
{'lastUpdated': Decimal('1739874353'), 'questionId': 'AT-0037-0', 'evaluationModelId': 'mara_evaluation_v0.1', 'text': 'How do your controls manage algorithmic trading risks? Detail your testing procedures, monitoring systems, and risk limits.', 'prepopulationModelId': 'mara_prepopulation_v0.1', 'sectionTitle': 'Algorithm Trading', 'sectionKey': 'algoTradingRisk'}
80808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080808080
I understand. I will analyze the provided content and answer the question about Algorithm Trading using the specified methodology and format. I will focus on providing strictly factual information without any subjective assessments or qualitative judgments.

<question>
1. How do your controls manage algorithmic trading risks? Detail your testing procedures, monitoring systems,

In [28]:
# Process each answer individually to handle per-question failures
for answer_data in answers:
    question_id = answer_data[1]
    print(question_id)
    question = next((q for q in questions if q['questionId'] == question_id), None)
    
    if question is None:
        print(f"Warning: No question found for ID {question_id}")
        continue
    
    # Get section_title from the question dictionary
    section_title = question.get('sectionTitle', 'Unknown Section')
    
    result = update_risk_assessment_with_answers(
        request_id=answer_data[0],
        firm_name=firm_name,
        question_id=question_id,
        question_text=question['text'],
        answer=answer_data[2],
        evidence=answer_data[3],
        section_title=section_title
    )

AT-0037-0
Updating DynamoDB with evidence: From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use your own in-house algorithms to execute orders?", NBEL Answer: "NO"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use the execution algos of your third-party brokers?", NBEL Answer: "NO TBC - Could happen if doing DMA on ICE/Euqrex"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use your own in-house investment decision making algos?", NBEL Answer: "NO"

From Risk Area "Algorithm Trading Risk", Question "How do your controls manage algorithmic trading risks?", Subquestion "Do you use third-party investment decision making algos?", NBEL Answer: "NO"
Current item before update: {'evidence': 'From Risk Area "Algorithm Trading Risk", Question "How do your 

In [45]:
# for answer_data in answers:
#     question_id = answer_data[1]
#     print(question_id)
#     question = next((q for q in questions if q['questionId'] == question_id), None)
#     print(question)

In [46]:
# print(answers[4])

In [38]:
# # Define the bucket name and object key
# bucket_name = 'batch-inference-input-experiment'
# object_key = 'input-data/output-data/trrx3arw4ygh/manifest.json.out'

# # Get the object from the S3 bucket
# response = s3.get_object(Bucket=bucket_name, Key=object_key)

# # Read the content of the object
# content = response['Body'].read().decode('utf-8')

# # Print the content
# print(content)